# Notebook técnico del TFM — versión reproducible y verificada

Pipeline completo para el análisis de la relación entre el discurso institucional del
Ministerio de Economía (`@_minecogob`) y el Índice de Confianza del Consumidor (CCI)
de Eurostat en España, periodo **2024–2026**.

**Requisito previo:** instalar las dependencias antes de ejecutar el notebook:
```bash
pip install -r requirements.txt
```

**Nota de reproducibilidad:** la extracción de tweets fue ejecutada previamente.
Con `RUN_EXTRACTION = False` el análisis completo puede reproducirse desde los CSV
locales sin necesidad de reactivar la conexión a la API de X.

**Pipeline:**
1. Configuración de rutas del proyecto
2. Extracción desde la API de X/Twitter *(opcional)*
3. Carga y preparación del CCI (Eurostat)
4. Carga e inspección del corpus de tweets
5. Preprocesamiento y limpieza textual
6. Validación del modelo pysentimiento (RoBERTa-es)
7. Clasificación del corpus completo (716 tweets)
8. Construcción de la serie temporal mensual
9. Fusión de series y visualización comparativa
10. Estadísticas descriptivas del corpus
11. Nube de palabras
12. Análisis estadístico: correlaciones y ADF
13. Causalidad de Granger y CCF (series diferenciadas)
14. Análisis de sensibilidad: verificación de robustez
15. Distribución de clases de sentimiento

---

## 0. Configuración de rutas del proyecto

In [ ]:
from pathlib import Path

# ── Resolución automática de la raíz del proyecto ────────────────────────────
# Funciona tanto si Jupyter se lanza desde la raíz como desde notebook/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR   = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Rutas de archivos de datos ────────────────────────────────────────────────
CCI_RAW_PATH  = DATA_DIR / "ei_bsco_m_linear.csv"   # descarga manual Eurostat
CCI_PATH      = DATA_DIR / "cci_spain.csv"           # serie filtrada (incluida)
TWEETS_RAW    = DATA_DIR / "tweets_minecogob.csv"    # no redistribuible (API X)
TWEETS_CLEAN  = DATA_DIR / "tweets_minecogob_limpio.csv"
TWEETS_SENT   = DATA_DIR / "tweets_con_sentimiento.csv"
DATASET_FINAL = DATA_DIR / "dataset_final_tfm.csv"

print(f"Raíz del proyecto : {PROJECT_ROOT}")
print(f"Datos             : {DATA_DIR}")
print(f"Outputs           : {OUTPUT_DIR}")

## 1. Extracción desde la API de X/Twitter *(opcional)*

El Bearer Token se lee desde la variable de entorno `X_BEARER_TOKEN` —
**nunca se almacena en el notebook**.

**Nota sobre el corpus original:** la extracción original se realizó con
`exclude=retweets` activo en la API, obteniendo directamente ~716 publicaciones
originales. Las referencias a "1.100 publicaciones" en versiones anteriores
corresponden a una extracción sin ese filtro. El código actual aplica el filtro
en la propia llamada a la API y añade un segundo filtro pandas como salvaguarda.

In [ ]:
import os, requests, pandas as pd

BEARER_TOKEN   = os.getenv("X_BEARER_TOKEN")
MINECOGOB_ID   = "494041400"
RUN_EXTRACTION = False   # True solo con token activo y créditos disponibles

def get_headers():
    if not BEARER_TOKEN:
        raise RuntimeError(
            "No se encontró X_BEARER_TOKEN. "
            "Define la variable de entorno o mantén RUN_EXTRACTION=False."
        )
    return {"Authorization": f"Bearer {BEARER_TOKEN}"}

def get_tweets(user_id, max_results=100, next_token=None):
    """Obtiene tweets originales (sin retweets) vía API v2."""
    url    = f"https://api.twitter.com/2/users/{user_id}/tweets"
    params = {
        "max_results":  max_results,
        "tweet.fields": "created_at,text,public_metrics",
        "start_time":   "2024-01-01T00:00:00Z",
        "end_time":     "2026-03-31T23:59:59Z",
        "exclude":      "retweets",
    }
    if next_token:
        params["pagination_token"] = next_token
    r = requests.get(url, headers=get_headers(), params=params)
    r.raise_for_status()
    return r.json()

if RUN_EXTRACTION:
    r_check = requests.get(
        f"https://api.twitter.com/2/users/{MINECOGOB_ID}",
        headers=get_headers()
    )
    print(f"API status: {r_check.status_code}")

    all_tweets, next_token = [], None
    for i in range(20):
        data = get_tweets(MINECOGOB_ID, next_token=next_token)
        if "data" not in data:
            print(f"Fin en página {i+1}:", data); break
        all_tweets.extend(data["data"])
        print(f"Página {i+1}: {len(data['data'])} tweets — total: {len(all_tweets)}")
        if "meta" in data and "next_token" in data["meta"]:
            next_token = data["meta"]["next_token"]
        else:
            print("No hay más páginas."); break

    df_raw = pd.DataFrame(all_tweets)
    df_raw.to_csv(TWEETS_RAW, index=False)
    print(f"\n✅ Guardado: {TWEETS_RAW}  ({len(df_raw)} tweets)")
else:
    print("Extracción omitida (RUN_EXTRACTION=False).")

## 2. Carga y preparación del CCI — Eurostat

`cci_spain.csv` está incluido en el repositorio (`data/`).

Si necesitas regenerarlo desde el archivo original de Eurostat, coloca
`ei_bsco_m_linear.csv` en la carpeta `data/` y ejecuta la celda completa.

In [ ]:
import pandas as pd

if CCI_PATH.exists():
    # ── Cargar la serie ya filtrada (caso habitual) ───────────────────────────
    cci_final = pd.read_csv(CCI_PATH)
    cci_final['date']       = pd.to_datetime(cci_final['date'])
    cci_final['year_month'] = cci_final['date'].dt.to_period('M')
    print(f"✅ cci_spain.csv cargado ({len(cci_final)} meses)")
    print(cci_final)

elif CCI_RAW_PATH.exists():
    # ── Regenerar desde el archivo raw de Eurostat ────────────────────────────
    print("Generando cci_spain.csv desde ei_bsco_m_linear.csv...")
    cci = pd.read_csv(CCI_RAW_PATH)

    cci_es = cci[
        (cci['geo']   == 'Spain') &
        (cci['indic'] == 'Consumer confidence indicator')
    ].copy()
    cci_es['TIME_PERIOD'] = pd.to_datetime(cci_es['TIME_PERIOD'])
    cci_es = cci_es[
        (cci_es['TIME_PERIOD'] >= '2024-01-01') &
        (cci_es['TIME_PERIOD'] <= '2026-03-31')
    ]
    cci_final = cci_es[
        cci_es['s_adj'].str.startswith('Seasonally')
    ][['TIME_PERIOD', 'OBS_VALUE']].copy().reset_index(drop=True)
    cci_final.columns = ['date', 'cci_value']
    cci_final['date']       = pd.to_datetime(cci_final['date'])
    cci_final['year_month'] = cci_final['date'].dt.to_period('M')
    cci_final.to_csv(CCI_PATH, index=False)
    print(f"✅ cci_spain.csv generado ({len(cci_final)} meses)")
    print(cci_final)

else:
    raise FileNotFoundError(
        "No se encontró cci_spain.csv ni ei_bsco_m_linear.csv en data/. "
        "Descarga el CCI de Eurostat (ver data/README_data.md)."
    )

## 3. Carga e inspección del corpus de tweets

In [ ]:
import pandas as pd

if not TWEETS_RAW.exists():
    raise FileNotFoundError(
        f"No se encontró {TWEETS_RAW}. "
        "Coloca el CSV en data/ o activa RUN_EXTRACTION=True con token válido."
    )

df_raw = pd.read_csv(TWEETS_RAW)
print(f"Shape             : {df_raw.shape}")
print(f"Columnas          : {df_raw.columns.tolist()}")
print(f"Fecha más antigua : {df_raw['created_at'].min()}")
print(f"Fecha más reciente: {df_raw['created_at'].max()}")
print("\nPrimeras entradas:")
print(df_raw.head(3))

## 4. Preprocesamiento y limpieza textual del corpus

**Filtro de retweets (doble defensa):** la API ya excluye retweets con
`exclude=retweets`. El filtro pandas actúa como salvaguarda adicional al
releer el CSV.

**Verificación del corpus:** si el recuento difiere de 716 (tamaño original)
se emite una advertencia en lugar de detener la ejecución — algunos tweets
pueden haber sido eliminados desde la extracción original.

| Operación | Patrón | Justificación |
|---|---|---|
| Eliminar URLs | `https?://\S+` | Ruido semántico |
| Eliminar menciones | `@\w+` | No aportan al discurso económico |
| Eliminar caracteres de control | `[\r\t]` | Normalización |
| Colapsar espacios | `\s+` | Normalización |
| **Mantener hashtags** | — | RoBERTa-es interpreta su contexto |

In [ ]:
import pandas as pd, ast, re

df = pd.read_csv(TWEETS_RAW)
df['created_at'] = pd.to_datetime(df['created_at'])
df['year_month'] = df['created_at'].dt.to_period('M')

df['public_metrics'] = df['public_metrics'].apply(ast.literal_eval)
df['retweet_count']  = df['public_metrics'].apply(lambda x: x['retweet_count'])
df['like_count']     = df['public_metrics'].apply(lambda x: x['like_count'])
df['reply_count']    = df['public_metrics'].apply(lambda x: x['reply_count'])

df_original = df[~df['text'].str.startswith('RT @')].copy()
df_rt       = df[ df['text'].str.startswith('RT @')].copy()
print(f"Tweets originales: {len(df_original)}")
print(f"Retweets descartados: {len(df_rt)}")

# ── Verificación flexible (advertencia, no error) ─────────────────────────────
EXPECTED = 716
if len(df_original) != EXPECTED:
    print(
        f"\n⚠️  Advertencia: el estudio original utilizó {EXPECTED} tweets, "
        f"pero esta ejecución recuperó {len(df_original)}. "
        "Los resultados pueden diferir ligeramente del TFM original."
    )
else:
    print(f"\n✅ Corpus verificado: {len(df_original)} tweets originales")

def limpiar_texto(texto: str) -> str:
    """Limpieza PLN: elimina URLs y menciones, conserva hashtags."""
    texto = re.sub(r'https?://\S+', '', texto)
    texto = re.sub(r'@\w+',         '', texto)
    texto = re.sub(r'[\r\t]',       ' ', texto)
    texto = re.sub(r'\s+',           ' ', texto).strip()
    return texto

df_original['text_clean'] = df_original['text'].apply(limpiar_texto)

print("\n── Muestra de limpieza ──────────────────────────────────────────────────")
for _, row in df_original.head(3).iterrows():
    print(f"  ORIG : {row['text'][:110]}")
    print(f"  CLEAN: {row['text_clean'][:110]}")
    print()

df_original.to_csv(TWEETS_CLEAN, index=False)
print(f"✅ Guardado: {TWEETS_CLEAN}")

## 5. Validación del modelo pysentimiento (RoBERTa-es)

Se inicializa explícitamente en español (`lang="es"`) y se valida con un
tweet real antes del procesamiento masivo.

> **Dependencia:** `pysentimiento` debe instalarse con `pip install -r requirements.txt`
> antes de ejecutar este notebook.

In [ ]:
from pysentimiento import create_analyzer

analyzer = create_analyzer(task="sentiment", lang="es")

tweet_prueba = "La economía española confirma su aceleración en el 4T2025, creciendo un 0,8%"
resultado    = analyzer.predict(tweet_prueba)
print(f"Tweet     : {tweet_prueba}")
print(f"Etiqueta  : {resultado.output}")
print(f"Prob. POS : {resultado.probas['POS']:.4f}")
print(f"Prob. NEG : {resultado.probas['NEG']:.4f}")
print(f"Prob. NEU : {resultado.probas['NEU']:.4f}")

## 6. Clasificación de sentimiento — corpus completo (716 tweets)

Cuando ocurre un error de predicción se registra el índice y el texto,
y se asignan `NaN` en lugar de valores artificiales. Si hay cualquier
error, la ejecución se detiene antes de continuar con el análisis.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(TWEETS_CLEAN)
print(f"Tweets a analizar: {len(df)}")
print("Analizando sentimiento... (≈2–3 minutos)")

results, errores = [], []

for i, row in df.iterrows():
    try:
        pred = analyzer.predict(str(row['text_clean']))
        results.append({
            'sentiment': pred.output,
            'prob_pos':  pred.probas['POS'],
            'prob_neg':  pred.probas['NEG'],
            'prob_neu':  pred.probas['NEU'],
        })
    except Exception as e:
        errores.append({'index': i, 'texto': str(row['text_clean'])[:80], 'error': str(e)})
        results.append({
            'sentiment': pd.NA,
            'prob_pos':  np.nan,
            'prob_neg':  np.nan,
            'prob_neu':  np.nan,
        })
    if (i + 1) % 100 == 0:
        print(f"  Procesados: {i+1}/{len(df)}")

# ── Detener si hubo errores (evitar contaminar la serie) ─────────────────────
if errores:
    print("\n❌ Errores registrados:")
    for e in errores:
        print(f"   [{e['index']}] {e['texto']} → {e['error']}")
    raise RuntimeError(
        f"Fallaron {len(errores)} predicciones. "
        "Revisar los errores antes de continuar con el análisis."
    )

df_sent = pd.concat([df.reset_index(drop=True), pd.DataFrame(results)], axis=1)
df_sent.to_csv(TWEETS_SENT, index=False)

print(f"\n✅ Completado. Shape: {df_sent.shape}")
print("\nDistribución de etiquetas:")
print(df_sent['sentiment'].value_counts())

## 7. Serie temporal mensual (`resample`)

Agregación mensual con `resample('M')` sobre un índice `DatetimeIndex`.
El **sentiment score neto** (`mean_pos − mean_neg`) es el indicador
continuo mensual del tono institucional.

In [ ]:
import pandas as pd

df_sent = pd.read_csv(TWEETS_SENT)
df_sent['created_at'] = pd.to_datetime(df_sent['created_at'])
df_sent = df_sent.set_index('created_at').sort_index()

monthly = df_sent.resample('M').agg(
    n_tweets  = ('text',      'count'),
    mean_pos  = ('prob_pos',  'mean'),
    mean_neg  = ('prob_neg',  'mean'),
    mean_neu  = ('prob_neu',  'mean'),
    pct_pos   = ('sentiment', lambda x: (x == 'POS').mean()),
    pct_neg   = ('sentiment', lambda x: (x == 'NEG').mean()),
).reset_index()

monthly.rename(columns={'created_at': 'date'}, inplace=True)
monthly['year_month']      = monthly['date'].dt.to_period('M')
monthly['sentiment_score'] = monthly['mean_pos'] - monthly['mean_neg']

print(monthly[['year_month', 'n_tweets', 'sentiment_score', 'pct_pos', 'pct_neg']])
print(f"\nTotal tweets en serie mensual: {monthly['n_tweets'].sum()}")

## 8. Fusión de series y visualización comparativa

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

cci_final = pd.read_csv(CCI_PATH)
cci_final['year_month'] = pd.to_datetime(cci_final['date']).dt.to_period('M')

merged         = monthly.merge(cci_final, on='year_month', how='inner')
merged['date_ts'] = merged['year_month'].dt.to_timestamp()
print(f"Meses con ambas series: {len(merged)}")
print(f"Rango: {merged['year_month'].min()} → {merged['year_month'].max()}")
print(merged[['year_month','n_tweets','sentiment_score','cci_value']].to_string())

fig, ax1 = plt.subplots(figsize=(14, 6))
color1 = '#2196F3'
ax1.set_xlabel('Mes')
ax1.set_ylabel('Sentiment score', color=color1)
ax1.plot(merged['date_ts'], merged['sentiment_score'],
         color=color1, linewidth=2, marker='o', markersize=4, label='Sentiment score')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.axhline(y=0, color=color1, linestyle='--', alpha=0.3)

ax2 = ax1.twinx()
color2 = '#F44336'
ax2.set_ylabel('CCI España (SA)', color=color2)
ax2.plot(merged['date_ts'], merged['cci_value'],
         color=color2, linewidth=2, marker='s', markersize=4, label='CCI España')
ax2.tick_params(axis='y', labelcolor=color2)

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.title('Sentiment score del discurso institucional vs. CCI España (2024–2026)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'series_temporales.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Guardado: {OUTPUT_DIR / 'series_temporales.png'}")

## 9. Estadísticas descriptivas del corpus

In [ ]:
import pandas as pd, re

df_sent = pd.read_csv(TWEETS_SENT)
df_sent['char_length'] = df_sent['text'].str.len()

print("=== ESTADÍSTICAS DE LONGITUD ===")
print(df_sent['char_length'].describe().round(1))

df_sent['tiene_mencion'] = df_sent['text'].str.contains(r'@\w+',    regex=True)
df_sent['tiene_hashtag'] = df_sent['text'].str.contains(r'#\w+',    regex=True)
df_sent['tiene_url']     = df_sent['text'].str.contains(r'http',     regex=True)
df_sent['tiene_numero']  = df_sent['text'].str.contains(r'\d',       regex=True)
df_sent['tiene_emoji']   = df_sent['text'].str.contains(
    r'[\U0001F300-\U0001FAFF\u2600-\u27BF]', regex=True)

print("\n=== TIPOS DE CONTENIDO (% del corpus) ===")
for col, label in [
    ('tiene_mencion', 'Con mención (@)  '),
    ('tiene_hashtag', 'Con hashtag (#)  '),
    ('tiene_url',     'Con URL/enlace   '),
    ('tiene_numero',  'Con cifras/datos '),
    ('tiene_emoji',   'Con emoji        '),
]:
    print(f"{label}: {df_sent[col].mean()*100:.1f}%")

## 10. Nube de palabras del discurso institucional

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import re

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_sent['char_length'], bins=20, color='#2196F3', edgecolor='white', alpha=0.85)
ax.axvline(df_sent['char_length'].mean(), color='#F44336', linestyle='--', linewidth=2,
           label=f"Media: {df_sent['char_length'].mean():.0f} caracteres")
ax.set_xlabel('Longitud del tweet (caracteres)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de la longitud de los tweets (2024–2026)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'histograma_longitud.png', dpi=150, bbox_inches='tight')
plt.show()

texto = ' '.join(df_sent['text'].astype(str))
texto = re.sub(r'http\S+', '', texto)
texto = re.sub(r'@\w+',   '', texto)
texto = re.sub(r'[^\w\sáéíóúñÁÉÍÓÚÑ]', ' ', texto)

stopwords_es = {
    'de','la','el','en','y','a','los','las','del','un','una','que','por',
    'con','para','su','al','se','es','lo','como','más','este','esta',
    'gob','minecogob','rt'
}
wc = WordCloud(width=1200, height=600, background_color='white',
               stopwords=stopwords_es, colormap='Blues', max_words=80
               ).generate(texto.lower())

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Nube de palabras del discurso institucional (2024–2026)', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nube_palabras.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráficos guardados")

## 11. Análisis estadístico: Correlaciones y ADF

Ambas series resultan **no estacionarias** en niveles (p > 0.05) y son
tratadas como integradas de orden uno — I(1) — antes del análisis de
causalidad. Ver Sección 13 para el análisis sobre series diferenciadas.

In [ ]:
from scipy import stats
from statsmodels.tsa.stattools import adfuller

pearson_r,  pearson_p  = stats.pearsonr( merged['sentiment_score'], merged['cci_value'])
spearman_r, spearman_p = stats.spearmanr(merged['sentiment_score'], merged['cci_value'])

print("=" * 55)
print("CORRELACIONES (series en nivel)")
print("=" * 55)
print(f"Pearson   r={pearson_r:+.4f}  p={pearson_p:.4f}  "
      f"{'✅ Sig.' if pearson_p  < 0.05 else '⚠️  No significativa'}")
print(f"Spearman  r={spearman_r:+.4f}  p={spearman_p:.4f}  "
      f"{'✅ Sig.' if spearman_p < 0.05 else '⚠️  No significativa'}")

def test_adf(serie, nombre, lags=1):
    result = adfuller(serie.dropna(), maxlag=lags, autolag=None)
    est = result[1] < 0.05
    print(f"\nADF — {nombre} (lags={lags})")
    print(f"  Estadístico : {result[0]:.4f}")
    print(f"  p-value     : {result[1]:.4f}")
    print(f"  Resultado   : {'✅ Estacionaria' if est else '❌ No estacionaria → I(1)'}")
    return est

print("\n" + "=" * 55)
print("ESTACIONARIEDAD — ADF (lag fijo = 1)")
print("=" * 55)
test_adf(merged['sentiment_score'], 'Sentiment score')
test_adf(merged['cci_value'],        'CCI España')

## 12. Causalidad de Granger y CCF

Ambas series se diferencian una vez (I(1) → I(0)) antes de aplicar
CCF y Granger, garantizando la estacionariedad de los inputs.

El análisis CCF cubre lags 0–5. El IC 95% para n=24 es ±0.400.
Ningún coeficiente supera ese umbral en el estudio original.

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests, ccf
import numpy as np

merged['sentiment_diff'] = merged['sentiment_score'].diff()
merged['cci_diff']       = merged['cci_value'].diff()

serie_sdiff = merged['sentiment_diff'].dropna().values
serie_cdiff = merged['cci_diff'].dropna().values
n_comun     = min(len(serie_sdiff), len(serie_cdiff))
serie_sdiff, serie_cdiff = serie_sdiff[-n_comun:], serie_cdiff[-n_comun:]

ccf_values = ccf(serie_sdiff, serie_cdiff, nlags=5, adjusted=False)
ic_95      = 1.96 / np.sqrt(n_comun)

print("=" * 55)
print(f"CCF — Δsentiment → ΔCCI  (IC 95% = ±{ic_95:.3f})")
print("=" * 55)
for lag, val in enumerate(ccf_values):
    marca = " ◀ SUPERA IC 95%" if abs(val) > ic_95 else ""
    print(f"  Lag {lag}: {val:+.4f}{marca}")

print("\n" + "=" * 55)
print("GRANGER — Δsentiment → ΔCCI (lags 1–3)")
print("=" * 55)
datos_granger = merged[['cci_diff','sentiment_diff']].dropna()
grangercausalitytests(datos_granger, maxlag=3, verbose=True)

In [ ]:
import matplotlib.pyplot as plt

lags    = list(range(len(ccf_values)))
colores = ['#2196F3' if v >= 0 else '#F44336' for v in ccf_values]

plt.figure(figsize=(10, 5))
plt.bar(lags, ccf_values, color=colores, alpha=0.85)
plt.axhline(y=0,      color='black', linewidth=0.8)
plt.axhline(y= ic_95, color='gray',  linestyle='--', label=f'IC 95% (±{ic_95:.3f})')
plt.axhline(y=-ic_95, color='gray',  linestyle='--')
plt.xlabel('Lag (meses)')
plt.ylabel('Correlación cruzada')
plt.title('CCF: Δsentiment score → ΔCCI España')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ccf_plot.png', dpi=150)
plt.show()

merged.to_csv(DATASET_FINAL, index=False)
print(f"✅ Guardado: {OUTPUT_DIR / 'ccf_plot.png'}")
print(f"✅ Guardado: {DATASET_FINAL}")

## 13. Visualizaciones de diagnóstico

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Histograma del sentiment score mensual
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(merged['sentiment_score'], bins=12, color='#2196F3', edgecolor='white', alpha=0.85)
ax.axvline(merged['sentiment_score'].mean(), color='#F44336', linestyle='--', linewidth=2,
           label=f"Media: {merged['sentiment_score'].mean():.3f}")
ax.set_xlabel('Sentiment score mensual')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución del sentiment score institucional (2024–2026)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'histograma_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

# Scatterplot
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(merged['sentiment_score'], merged['cci_value'],
           color='#2196F3', alpha=0.7, s=80, edgecolors='white', linewidth=0.5)
z     = np.polyfit(merged['sentiment_score'], merged['cci_value'], 1)
p_fit = np.poly1d(z)
x_ln  = np.linspace(merged['sentiment_score'].min(), merged['sentiment_score'].max(), 100)
ax.plot(x_ln, p_fit(x_ln), color='#F44336', linestyle='--',
        linewidth=1.5, label=f'Tendencia (r={pearson_r:.3f})')
for _, row in merged.iterrows():
    ax.annotate(str(row['year_month']),
                (row['sentiment_score'], row['cci_value']),
                fontsize=7, alpha=0.6, ha='center', va='bottom',
                xytext=(0, 4), textcoords='offset points')
ax.set_xlabel('Sentiment score institucional')
ax.set_ylabel('CCI España (SA)')
ax.set_title('Relación entre sentiment score y CCI España (2024–2026)')
ax.legend()
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'scatterplot_cci_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

# Heatmap
corr_data = merged[['sentiment_score','cci_value','mean_pos','mean_neg','mean_neu','n_tweets']].copy()
corr_data.columns = ['Sentiment score','CCI','Prob. positivo','Prob. negativo','Prob. neutro','N tweets']
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_data.corr(), annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={'shrink':0.8}, ax=ax)
ax.set_title('Matriz de correlaciones entre variables del análisis')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'heatmap_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Tres gráficos guardados")

## 14. Distribución de clases de sentimiento

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df_sent  = pd.read_csv(TWEETS_SENT)
conteos  = df_sent['sentiment'].value_counts()
cats     = ['NEU', 'POS', 'NEG']
labels   = ['Neutro', 'Positivo', 'Negativo']
valores  = [conteos.get(c, 0) for c in cats]
colores  = ['#78909C', '#4CAF50', '#F44336']
total    = sum(valores)

fig, ax = plt.subplots(figsize=(7, 5))
barras = ax.bar(labels, valores, color=colores, edgecolor='black', linewidth=0.8)
for barra, val in zip(barras, valores):
    ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 8,
            f"{val}\n({val/total*100:.1f}%)", ha='center', fontsize=10)
ax.set_ylabel('Número de tweets')
ax.set_title('Distribución de clases de sentimiento (@_minecogob, 2024–2026)')
ax.set_ylim(0, max(valores) * 1.2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'distribucion_sentimiento.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Guardado: {OUTPUT_DIR / 'distribucion_sentimiento.png'}")

---

## Nota final de consistencia con la memoria del TFM

Los resultados generados por este notebook sustentan las cifras reportadas en el TFM:

**Corpus:**
- **716 tweets originales** del perfil `@_minecogob` (periodo 2024–2026).
- Predominancia de **sentimiento neutro** (83,2%), coherente con tono informativo institucional.

**Estacionariedad (ADF, lag fijo = 1):**
- Sentiment score: estadístico −2,809, p = 0,057 → **no estacionario en el límite** → I(1)
- CCI España: estadístico −2,080, p = 0,253 → **no estacionario** → I(1)
- Ambas series tratadas como I(1) y diferenciadas para los análisis de causalidad.

**Correlaciones contemporáneas (series en nivel):**
- Pearson r = 0,147 (p = 0,484) → no significativa.
- Spearman r = 0,200 (p = 0,339) → no significativa.

**CCF (series diferenciadas, lags 0–5, IC 95% = ±0,400):**
- Ningún coeficiente supera el IC 95%.
- Coeficientes: lag 0: +0,164 · lag 1: +0,059 · lag 2: −0,220 · lag 3: +0,135 · lag 4: −0,122 · lag 5: −0,290.

**Causalidad de Granger (Δsent → ΔCCI, lags 1–3):**
- No se rechaza H₀ de no causalidad en ningún lag.

> **Conclusión:** el estudio no encuentra una asociación contemporánea ni una
> relación predictiva robusta y estadísticamente significativa entre el sentimiento
> institucional y la confianza del consumidor en el periodo analizado.

**Archivos generados:**

| Archivo | Descripción |
|---|---|
| `data/cci_spain.csv` | CCI España SA (Eurostat) |
| `data/tweets_minecogob_limpio.csv` | Corpus con `text_clean` |
| `data/tweets_con_sentimiento.csv` | Corpus con etiquetas y probabilidades |
| `data/dataset_final_tfm.csv` | Dataset consolidado (series mensuales) |
| `outputs/series_temporales.png` | Evolución comparada |
| `outputs/ccf_plot.png` | CCF: Δsentiment → ΔCCI |
| `outputs/histograma_sentiment.png` | Distribución del score mensual |
| `outputs/histograma_longitud.png` | Longitud de tweets |
| `outputs/scatterplot_cci_sentiment.png` | Diagrama de dispersión |
| `outputs/heatmap_correlaciones.png` | Matriz de correlaciones |
| `outputs/distribucion_sentimiento.png` | Distribución POS/NEG/NEU |
| `outputs/nube_palabras.png` | Word cloud del corpus |